In [ ]:
!nvidia-smi


Thu Oct  8 07:22:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from numba import cuda
device = cuda.get_current_device()
print("Device Name:", device.name)
print("Compute Capability:", device.compute_capability)
print("SMs:", device.MULTIPROCESSOR_COUNT)
print("Max Threads Per Block:", device.MAX_THREADS_PER_BLOCK)
print("Max Block Dims:", device.MAX_BLOCK_DIM_X, device.MAX_BLOCK_DIM_Y, device.MAX_BLOCK_DIM_Z)
print("Warp Size:", device.WARP_SIZE)

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
SMs: 40
Max Threads Per Block: 1024
Max Block Dims: 1024 1024 64
Warp Size: 32


In [ ]:
import numpy as np, time
from numba import cuda

def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [ ]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [ ]:
for N in [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]:
    print(N, benchmark_run(N))

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


100 (0.06132099997557816, 1.430560000017067, 0.08814199998141703)
1000 (0.3090499999416352, 0.6463870000743555, 0.056051000001389184)
10000 (2.4713829999427617, 0.5475949999436125, 0.05698699999356904)
100000 (24.07341200000701, 0.7003609999856053, 0.07675699998799246)
1000000 (252.0274230000723, 4.58471499996449, 0.12548200004403043)
5000000 (nan, 11.948598000003585, 0.2608300000019881)
10000000 (nan, 24.97517900008006, 0.46609499997884996)


In [ ]:
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    d_arr[idx] = 999.0   # нет проверки idx < n

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

print("Launching defective kernel...")
defective_kernel[4, 256](d_arr)
cuda.synchronize()
h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

Launching defective kernel...
Elements processed: 1000. Last element: 999.0


In [ ]:
import numpy as np, time

arr = np.ones(50_000, dtype=np.float32)
d_arr = cuda.to_device(arr)
tpb = 256
bpg = (arr.size + tpb - 1) // tpb

# 5 попыток naive
for trial in range(1, 6):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    naive_sum_kernel[bpg, tpb](d_arr, d_total)
    cuda.synchronize()
    res = d_total.copy_to_host()[0]
    print(f"Trial {trial}: {res}  lost={50000.0 - res}")

d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
atomic_sum_kernel[bpg, tpb](d_arr, d_total)
cuda.synchronize()
print("Atomic:", d_total.copy_to_host()[0])

t0 = time.perf_counter()
d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
naive_sum_kernel[bpg, tpb](d_arr, d_total)
cuda.synchronize()
naive_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
atomic_sum_kernel[bpg, tpb](d_arr, d_total)
cuda.synchronize()
atomic_ms = (time.perf_counter() - t0) * 1000

print(f"Naive: {naive_ms:.4f} ms | Atomic: {atomic_ms:.4f} ms")

Trial 1: 2.0  lost=49998.0
Trial 2: 2.0  lost=49998.0
Trial 3: 2.0  lost=49998.0
Trial 4: 2.0  lost=49998.0
Trial 5: 2.0  lost=49998.0
Atomic: 50000.0
Naive: 0.7701 ms | Atomic: 0.7519 ms


In [ ]:
import math
import numpy as np
from numba import cuda

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1],
)

radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()

print(f"Generated 2D matrix shape: {result_canvas.shape}. Center value: {result_canvas[512, 512]:.3f}")
print("Corner (0,0):", result_canvas[0, 0])
print("Blocks:", blocks_2d, "Total:", blocks_2d[0] * blocks_2d[1])

Generated 2D matrix shape: (1024, 1024). Center value: 1.000
Corner (0,0): 0.0
Blocks: (64, 64) Total: 4096


In [ ]:
import hashlib
from numba import cuda

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))

    name = cuda.get_current_device().name
    h.update(name if isinstance(name, bytes) else name.encode('utf-8'))

    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()

    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

STUDENT_ID = "230103172"
CROSSOVER_N = 10000
generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: BEBE56352B96113C


'BEBE56352B96113C'